In [ ]:
import pandas as pd

In [ ]:
pos = pd.read_csv("D:/AI/HomeCredit/Dataset/POS_CASH_balance.csv")

In [ ]:
pos_agg_prev = pos.groupby('SK_ID_PREV').agg(
    pos_months_count = ('MONTHS_BALANCE', 'count'),
    pos_months_min = ('MONTHS_BALANCE', 'min'),
    pos_months_max = ('MONTHS_BALANCE', 'max'),
    pos_cnt_instalment_mean = ('CNT_INSTALMENT', 'mean'),
    pos_cnt_instalment_future_mean = ('CNT_INSTALMENT_FUTURE', 'mean'),
    pos_dpd_max = ('SK_DPD', 'max'),
    pos_dpd_mean = ('SK_DPD', 'mean'),
    pos_dpd_def_max = ('SK_DPD_DEF', 'max'),
    pos_overdue_months = ('SK_DPD', lambda x: (x>0).sum()),
    pos_severe_overdue_months = ('SK_DPD_DEF', lambda x: (x>0).sum())
).reset_index()

In [ ]:
pos = pd.get_dummies(
    pos,
    columns=['NAME_CONTRACT_STATUS'],
    prefix='POS_STATUS',
    dummy_na=False
)
pos_final = pos.merge(pos_agg_prev, on='SK_ID_PREV', how='left')
pos_final = pos_final.groupby('SK_ID_CURR').agg(['mean', 'sum', 'max'])
pos_final.columns = ['_'.join(col).upper() for col in pos_final.columns]
pos_final.reset_index(inplace=True)
pos_final

In [ ]:
inst = pd.read_csv("D:/AI/HomeCredit/Dataset/installments_payments.csv")

In [ ]:
inst['PAYMENT_DELAY'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1e-6)

In [ ]:
inst_agg_prev = inst.groupby('SK_ID_PREV').agg(
    inst_count = ('NUM_INSTALMENT_NUMBER', 'count'),
    payment_delay_mean = ('PAYMENT_DELAY', 'mean'),
    payment_delay_max = ('PAYMENT_DELAY', 'max'),
    payment_ratio_mean = ('PAYMENT_RATIO', 'mean'),
    payment_ratio_min = ('PAYMENT_RATIO', 'min'),
    payment_diff_sum = ('PAYMENT_DIFF', 'sum'),
    late_payments = ('PAYMENT_DELAY', lambda x: (x>0).sum())
).reset_index()

In [ ]:
inst_final = inst.merge(inst_agg_prev, on='SK_ID_PREV', how='left')
inst_final = inst_final.groupby('SK_ID_CURR').agg(['mean','sum','max'])
inst_final.columns = ['_'.join(col).upper() for col in inst_final.columns]
inst_final.reset_index()

In [ ]:
cc = pd.read_csv("D:/AI/HomeCredit/Dataset/credit_card_balance.csv")
cc.head()

In [ ]:
cc['UTILIZATION'] = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL']
cc['PAYMENT_RATIO'] = cc['AMT_PAYMENT_TOTAL_CURRENT'] / cc['AMT_TOTAL_RECEIVABLE']

In [ ]:
cc_agg_prev = cc.groupby('SK_ID_PREV').agg(
    cc_months_count = ('MONTHS_BALANCE','count'),
    utilization_mean = ('UTILIZATION', 'mean'),
    utilization_max = ('UTILIZATION', 'max'),
    payment_ratio_mean = ('PAYMENT_RATIO', 'mean'),
    cc_dpd_max = ('SK_DPD', 'max'),
    cc_dpd_def_max = ('SK_DPD_DEF', 'max'),
    total_drawings = ('AMT_DRAWINGS_CURRENT' , 'sum')
).reset_index()

In [ ]:
cc_final = cc.merge(cc_agg_prev , on ='SK_ID_PREV', how = 'left')
num_cols = cc_final.select_dtypes(include=['int64','float64']).columns
cc_final = cc_final.groupby('SK_ID_CURR')[num_cols]. agg(['mean','sum','max'])
cc_final.columns = ['_'.join(col).upper()for col in cc_final.columns]
cc_final.reset_index(inplace=True)

In [ ]:
cc_final.head()

In [ ]:
pos_inst_cc_merged = pos_final.merge(cc_final, on = 'SK_ID_CURR' , how='left')
pos_inst_cc_merged = pos_inst_cc_merged.merge(inst_final, on = 'SK_ID_CURR', how='left')

In [ ]:
pos_inst_cc_merged.head()

In [ ]:
pos_inst_cc_merged.to_csv("pos_inst_cc_final.csv", index = False)